# __Demo: Generating Fake Images with Generative Adversarial Networks (GANs)__
## What we’re doing
We build the Generator and Discriminator as small CNNs. Convolutions learn spatial structure—edges, strokes, and layout—so the Generator must produce digit-like shapes to fool the Discriminator. A plain MLP tends to latch onto global intensity instead of structure. On MNIST, with its large uniform background, that can drive the GAN toward a trivial “constant image” (all dark or all bright) that briefly fools a weak Discriminator. The CNN setup removes that loophole: the Discriminator judges local features, not just brightness, and the Generator learns to draw actual digits.

### FAQ
<ul>
<li>Why CNNs? They capture local patterns (strokes, edges).</li>
<li>Why not MLPs? They overvalue average brightness.</li>
<li>Failure mode avoided: collapsing to near-constant images.</li>
<li>Net effect: the model learns shapes, not shortcuts.</li>
</ul>

# __Steps to Perform__

Step 1: Import the Necessary Libraries

Step 2: Load and Preprocess the Data

Step 3: Build the Generator and Discriminator

Step 4: Compile the Models

Step 5: Train the Models

Step 6: Execute the Training

Step 7: Generate New Images and Evaluate the Model's Performance



# __Step 1: Import the Necessary Libraries__

In [1]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
import torch, torch.nn as nn
from pathlib import Path
from torchvision.utils import save_image
from torch.utils.data import DataLoader, TensorDataset

# Define home and output dir
cwd = Path('.').resolve()
if str(cwd).split("/")[-1] == 'notebooks':
    HOME = cwd.parent
else:
    HOME = cwd.parent.parent
OUTDIR = HOME / 'results' / 'GANs_CNN_vs_MLP'


# __Step 2: Load and Preprocess the Data__

- Load the MNIST dataset and preprocess it.
- Preprocessing involves normalizing the data that can improve models' performance.

In [2]:
# Load the MNIST dataset
def load_mnist():
    (x, _), _ = tf.keras.datasets.mnist.load_data()
    x = (x.astype("float32") - 127.5) / 127.5   # convert data from [0,255] to [-1,1]
    x = x[..., None]                             # (N,28,28,1)
    x = torch.as_tensor(x, dtype=torch.float32).unsqueeze(1)  # [N,1,28,28]
    return x


# __Step 3: Build the Generator and Discriminator__

- Define the generator and discriminator models.
- Generator takes a random noise vector as input and outputs an image.
- Discriminator takes an image as input and outputs the probability of the image being real.

In [3]:
class GenMLP(nn.Module):
    def __init__(self, latent_dim=100, img_ch=1, img_sz=28):
        super().__init__()
        self.img_ch, self.img_sz = img_ch, img_sz
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.ReLU(True),
            nn.Linear(1024, img_ch*img_sz*img_sz), nn.Tanh(),
        )
    def forward(self, z):
        x = self.net(z)
        return x.view(z.size(0), self.img_ch, self.img_sz, self.img_sz)

class DiscMLP(nn.Module):
    def __init__(self, img_ch=1, img_sz=28):
        super().__init__()
        in_dim = img_ch*img_sz*img_sz
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 512), nn.LeakyReLU(0.2, True),
            nn.Linear(512, 256),    nn.LeakyReLU(0.2, True),
            nn.Linear(256, 1),
        )
    def forward(self, x): return self.net(x).view(-1)  # logits

class GenConv(nn.Module):
    def __init__(self, latent_dim=100, img_ch=1, base=64):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, (base*4)*7*7, bias=False),
            nn.BatchNorm1d((base*4)*7*7), nn.ReLU(True),
        )
        self.up = nn.Sequential(
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1, bias=False),  # 7→14
            nn.BatchNorm2d(base*2), nn.ReLU(True),
            nn.ConvTranspose2d(base*2, img_ch, 4, 2, 1, bias=True),   # 14→28
            nn.Tanh(),
        )
    def forward(self, z):
        x = self.fc(z).view(z.size(0), -1, 7, 7)
        return self.up(x)

class DiscConv(nn.Module):
    def __init__(self, img_ch=1, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_ch, base, 4, 2, 1, bias=True),               # 28→14
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base*2, 4, 2, 1, bias=False),              # 14→7
            nn.BatchNorm2d(base*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*2, 1, 7, 1, 0, bias=True),                  # 7→1
        )
    def forward(self, x): return self.net(x).view(-1)  # logits


# __Step 4: Compile the Models__

- Compile the models, which involves defining the loss function and the optimizer.
- The loss function evaluates the model's performance, while the optimizer aims to minimize the loss.

In [4]:
# --- Toggle + init ---
def make_models(use_conv=True, latent_dim=100, img_ch=1, img_sz=28):
    if use_conv:
        G, D = GenConv(latent_dim, img_ch), DiscConv(img_ch)
    else:
        G, D = GenMLP(latent_dim, img_ch, img_sz), DiscMLP(img_ch, img_sz)
    for m in (G, D):
        for layer in m.modules():
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.normal_(layer.weight, 0.0, 0.02)
                if getattr(layer, "bias", None) is not None:
                    nn.init.zeros_(layer.bias)
    return G, D

use_conv = True  # flip to False for MLP
G, D = make_models(use_conv, latent_dim=100, img_ch=1, img_sz=28)

# __Step 5: Train the Models__

- Train the model, which involves feeding data into the models and adjusting the weights of the models based on the output.
- The primary aim is for the generator to create images indistinguishable from real images by the discriminator.

In [5]:
# --- Tiny training stub (same loop; only architecture toggles) ---
def train_one_epoch(G, D, dataloader, z_dim=100, device="cpu"):
    G.train(); D.train()
    bce = nn.BCEWithLogitsLoss()
    optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    for real,_ in dataloader:
        real = real.to(device)
        # scale to [-1,1] for Tanh generator
        real = real*2 - 1

        B = real.size(0)
        z = torch.randn(B, z_dim, device=device)

        # --- Discriminator step ---
        D.zero_grad(set_to_none=True)
        logits_real = D(real)
        fake = G(z).detach()
        logits_fake = D(fake)
        lossD = bce(logits_real, torch.full_like(logits_real, 0.9)) + \
                bce(logits_fake, torch.zeros_like(logits_fake))
        lossD.backward(); optD.step()

        # --- Generator step ---
        G.zero_grad(set_to_none=True)
        z = torch.randn(B, z_dim, device=device)
        fake = G(z)
        logits_fake = D(fake)
        lossG = bce(logits_fake, torch.ones_like(logits_fake))
        lossG.backward(); optG.step()



# __Step 6: Execute the Training__

In [8]:

# --- one-time helpers ---
def make_optimizers(G, D, lr=2e-4):
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    return optG, optD

def d_step(D, G, real, bce, optD, z_dim, device, B, inst_noise=0.0):
    z = torch.randn(B, z_dim, device=device)
    with torch.no_grad():
        fake = G(z)
    x_real = real + inst_noise * torch.randn_like(real) if inst_noise else real
    x_fake = fake + inst_noise * torch.randn_like(fake) if inst_noise else fake
    logits_real = D(x_real)
    logits_fake = D(x_fake)
    lossD = (
        bce(logits_real, torch.full_like(logits_real, 0.9)) +  # label smoothing
        bce(logits_fake, torch.zeros_like(logits_fake))
    )
    optD.zero_grad(set_to_none=True); lossD.backward(); optD.step()
    return float(lossD.item())

def g_step(D, G, bce, optG, z_dim, device, B):
    z = torch.randn(B, z_dim, device=device)
    fake = G(z)
    logits_fake = D(fake)
    lossG = bce(logits_fake, torch.ones_like(logits_fake))
    optG.zero_grad(set_to_none=True); lossG.backward(); optG.step()
    return float(lossG.item())

def save_sample(G, z_fixed, outdir, epoch, value_range=(-1,1)):
    G.eval()
    with torch.no_grad():
        imgs = G(z_fixed).clamp(min=value_range[0], max=value_range[1])
    # map from [-1,1] → [0,1] for saving
    imgs = (imgs - value_range[0]) / (value_range[1]-value_range[0])
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)
    save_image(imgs, out / f"samples_epoch_{epoch:03d}.png", nrow=8)
    G.train()


def preprocess_batch(x, device):
    # unwrap (x, y)
    if isinstance(x, (list, tuple)): x = x[0]

    # tf.Tensor / np -> torch
    if hasattr(x, "numpy"): x = torch.from_numpy(x.numpy())
    elif not isinstance(x, torch.Tensor): x = torch.tensor(np.array(x))

    x = x.float()

    # squeeze stray trailing 1: [B,1,28,28,1] -> [B,1,28,28]
    if x.ndim == 5 and x.shape[-1] == 1: x = x.squeeze(-1)

    # NHWC -> NCHW if needed
    if x.ndim == 4 and x.shape[1] not in (1,3) and x.shape[-1] in (1,3):
        x = x.permute(0, 3, 1, 2).contiguous()

    # scale once for tanh: if [0,255] -> [0,1]; if [0,1] -> [-1,1]; if already [-1,1] do nothing
    x_min, x_max = float(x.min()), float(x.max())
    if x_max > 1.5:          # likely 0..255
        x = x / 255.0
        x = x * 2 - 1
    elif 0.0 <= x_min and x_max <= 1.0:  # 0..1
        x = x * 2 - 1
    # else assume already in [-1,1]

    return x.to(device)



def to_torch_nchw(x, device):
    # unwrap (x, y)
    if isinstance(x, (list, tuple)):
        x = x[0]

    # tf.Tensor → np → torch
    if hasattr(x, "numpy"):
        x = torch.from_numpy(x.numpy())
    elif not isinstance(x, torch.Tensor):
        x = torch.tensor(np.array(x))

    x = x.float()

    # squeeze any trailing singleton (… , 1)
    if x.ndim >= 3 and x.shape[-1] == 1:
        x = x.squeeze(-1)  # e.g., [B,28,28,1] -> [B,28,28]

    # ensure 4D NCHW
    if x.ndim == 3:                  # [B,H,W] -> add channel
        x = x.unsqueeze(1)           # [B,1,H,W]
    elif x.ndim == 4:
        # If channel is last (NHWC), move it to first: [B,H,W,C] -> [B,C,H,W]
        if x.shape[1] not in (1, 3) and x.shape[-1] in (1, 3):
            x = x.permute(0, 3, 1, 2).contiguous()

    return x.to(device)

# --- main training loop ---
def train(ds, steps, epochs, batch, G, D, z_dim=100, device="cpu",
          sample_every=5, outdir="results/gan"):
    import torch, torch.nn as nn
    from torchvision.utils import save_image
    G.to(device); D.to(device); G.train(); D.train()
    bce = nn.BCEWithLogitsLoss()
    optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
    optG = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5,0.999))
    z_fixed = torch.randn(64, z_dim, device=device)

    for ep in range(1, epochs+1):
        it = iter(ds)               # new iterator each epoch (shuffle if loader supports it)
        lossD_running = lossG_running = 0.0

        for _ in range(steps):
            real = preprocess_batch(next(it), device)  # <- advances ONCE
            B = real.size(0)

            # D twice, G once (helps stability)
            lossD_running += d_step(D, G, real, bce, optD, z_dim, device, B, inst_noise=0.05)
            lossD_running += d_step(D, G, real, bce, optD, z_dim, device, B, inst_noise=0.05)
            lossG_running += g_step(D, G, bce, optG, z_dim, device, B)

        print(f"[epoch {ep:03d}] D: {lossD_running/(2*steps):.4f} | G: {lossG_running/steps:.4f}")

        if ep % sample_every == 0 or ep in (1, epochs):
            with torch.no_grad():
                imgs = G(z_fixed).clamp(-1,1)
            save_image((imgs+1)/2, f"{outdir}/samples_epoch_{ep:03d}.png", nrow=8)

    # final sample
    save_sample(G, z_fixed, outdir, epochs)
    # save model
    torch.save(G.state_dict(), outdir / "model.pt")


In [10]:
# config / toggle
USE_CONV   = True   # False → MLP
LATENT_DIM = 100
IMG_CH     = 1
IMG_SZ     = 28
BATCH_SIZE = 128
EPOCHS = 50

# load data
X = load_mnist()                   # [N,28,28] in [0,1], TF or np
# then make a torch DataLoader
ds = DataLoader(TensorDataset(X), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
steps_per_epoch = len(ds)

# models
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
G, D = make_models(use_conv=USE_CONV, latent_dim=LATENT_DIM, img_ch=IMG_CH, img_sz=IMG_SZ)

# train
train(ds=ds,
      steps=steps_per_epoch,
      epochs=EPOCHS,
      batch=BATCH_SIZE,
      G=G, D=D,
      z_dim=LATENT_DIM,
      device=device,
      sample_every=5,
      outdir=OUTDIR / "CNN" if USE_CONV else OUTDIR / "MLP")


[epoch 001] D: 0.4252 | G: 4.2973
[epoch 002] D: 0.4501 | G: 3.3803
[epoch 003] D: 0.4900 | G: 3.3628
[epoch 004] D: 0.5253 | G: 3.0906
[epoch 005] D: 0.5476 | G: 3.0191
[epoch 006] D: 0.5649 | G: 2.9059
[epoch 007] D: 0.5932 | G: 2.8516
[epoch 008] D: 0.5749 | G: 2.8140
[epoch 009] D: 0.5861 | G: 2.8183
[epoch 010] D: 0.5784 | G: 2.8490
[epoch 011] D: 0.5770 | G: 2.8902
[epoch 012] D: 0.5756 | G: 2.8811
[epoch 013] D: 0.5617 | G: 2.9890
[epoch 014] D: 0.5851 | G: 3.0060
[epoch 015] D: 0.5587 | G: 3.0566
[epoch 016] D: 0.5828 | G: 3.0290
[epoch 017] D: 0.5720 | G: 3.1650
[epoch 018] D: 0.5756 | G: 2.9693
[epoch 019] D: 0.5590 | G: 3.1241
[epoch 020] D: 0.5695 | G: 3.1230
[epoch 021] D: 0.5614 | G: 3.1931
[epoch 022] D: 0.5539 | G: 3.1474
[epoch 023] D: 0.5924 | G: 3.1236
[epoch 024] D: 0.5530 | G: 3.1501
[epoch 025] D: 0.5879 | G: 3.1609
[epoch 026] D: 0.5703 | G: 3.0146
[epoch 027] D: 0.5905 | G: 3.1461
[epoch 028] D: 0.5714 | G: 3.0887
[epoch 029] D: 0.5670 | G: 3.1375
[epoch 030] D:

**Notes:**
- Epochs parameter determines how many times the learning algorithm will work through the entire training dataset.
- The `batch_size` is the number of samples that will be propagated through the network at a time.

# __Step 7: Generate New Images and Evaluate the Model's Performance__

- Generate new images and evaluate the performance of the GAN.
- Generate a random noise vector and feed it into the trained generator to create new images.

In [ ]:
# generates a 10x10 grid of images based on the model_type (CNN or MLP)
def gen_images(model_type: str):
    ckpt = torch.load(OUTDIR / model_type / "model.pt", map_location="cpu")
    if model_type == 'CNN':
        G = GenConv(LATENT_DIM, IMG_CH)
    else:
        G = GenMLP(LATENT_DIM, IMG_CH, IMG_SZ)
    G.load_state_dict(ckpt["model_state"])
    N = 100

    device = next(G.parameters()).device

    # latent: (N, LATENT_DIM)
    z = torch.from_numpy(np.random.randn(N, LATENT_DIM).astype(np.float32)).to(device)

    G.eval()
    with torch.no_grad():
        imgs = G(z).clamp(-1, 1)            # [N, C, H, W] in [-1,1]
        imgs = (imgs + 1) / 2

    # Quick diagnostics (useful if it's still dark)
    print("min/max/mean:", float(imgs.min()), float(imgs.max()), float(imgs.mean()))
    imgs = imgs.permute(0, 2, 3, 1).cpu().numpy()

    # Visualize a 10x10 grid
    h = int(np.sqrt(N))
    w = N//h
    if N%h > 0:
        w += 1
    plt.figure(figsize=(w,h))
    for i in range(N):
        plt.subplot(w, h, i+1)
        plt.imshow(imgs[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
        plt.axis('off')
    plt.tight_layout()
    plt.show()



In [ ]:
gen_images("CNN")

IsADirectoryError: [Errno 21] Is a directory: '/Users/douglasdaly/Documents/GitHub/Generative-AI/results/GANs_CNN_vs_MLP/CNN'

In [ ]:
gen_images("MLP")

- The resulting plot shows the images generated by the GAN model.


# __Conclusion__

In this demo, you have successfully implemented a GAN to generate images resembling handwritten digits, focusing on the MNIST dataset. The process involved constructing and training a generator and a discriminator. By toggling between using a CNN vs a MLP we see why CNNs are necessary for image generation and recognition.